# LOST - Continuous Mountain Car

In [1]:
import numpy as np
import gymnasium as gym
import random 

In [2]:
# Cambiar render_mode a rgb_array para entrenar/testear sino human
env_id = 'MountainCarContinuous-v0'
env = gym.make(env_id, render_mode='rgb_array')

In [ ]:
from q_learning_agent import QLearningAgent

Observation Space

In [3]:
env.observation_space

Box([-1.2  -0.07], [0.6  0.07], (2,), float32)

Action Space

In [4]:
env.action_space

Box(-1.0, 1.0, (1,), float32)

Discretización de los estados

In [ ]:
x_space = np.linspace(-1.2, 0.6, 100)
vel_space = np.linspace(-0.07, 0.07, 100)
x_space

Obtener el estado a partir de la observación

In [ ]:
def get_state(obs):
    x, vel = obs
    x_bin = np.digitize(x, x_space)
    vel_bin = np.digitize(vel, vel_space)
    return x_bin, vel_bin

In [ ]:
state = get_state(np.array([-0.4, 0.02]))
state

Discretización de las acciones

In [ ]:
actions = list(np.linspace(-1, 1, 10))
actions

In [ ]:
def get_sample_action():
    return random.choice(actions)

Inicilización de la tabla Q

In [ ]:
Q = np.zeros((len(x_space) + 1, len(vel_space) + 1, len(actions)))
Q

Obtención de la acción a partir de la tabla Q

In [ ]:
def optimal_policy(state, Q):
    action = actions[np.argmax(Q[state])]
    return action

Epsilon-Greedy Policy

In [ ]:
def epsilon_greedy_policy(state, Q, epsilon=0.1):
    explore = np.random.binomial(1, epsilon)
    if explore:
        action = get_sample_action()
    else:
        action = optimal_policy(state, Q)
        
    return action

# Prueba renderizada

In [3]:
# Testear el agente entrenado (sin exploración)
print("\nTesteando agente entrenado...")
from dyna_q_agent import DynaQAgent
agent = DynaQAgent(x_bins=100, vel_bins=100, action_bins=20)
agent.load_model('dyna_agent_trained.npy')

env = gym.make('MountainCarContinuous-v0', render_mode='human')
test_rewards, test_steps = agent.test_agent(env, episodes=3, render=True)
env.close()

print(f"\nPromedio de rewards en testing: {np.mean(test_rewards):.2f}")
print(f"Promedio de steps en testing: {np.mean(test_steps):.2f}")


Testeando agente entrenado...
Modelo cargado desde dyna_agent_trained.npy (Q shape: (101, 101, 20))
Test Episode 1 - Reward: 94.43, Steps: 335, Success: 1
Test Episode 2 - Reward: 94.56, Steps: 119, Success: 1
Test Episode 3 - Reward: 94.64, Steps: 115, Success: 1


ValueError: too many values to unpack (expected 2)

# Q-Learning Agent

In [ ]:
from q_learning_agent import QLearningAgent

In [ ]:
from q_experiments_manager import ExperimentManager

# 1. Crear el entorno
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')

# 2. Inicializar el mánager de experimentos
manager = ExperimentManager()

In [ ]:
print("Entrenando agente Q-Learning...")
id_1 = "8k_episodes_eps015"
manager.train_run(
    env=env, 
    run_id=id_1,
    x_bins=100,         
    vel_bins=100,       
    action_bins=20,    
    episodes=8000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.15,
    max_steps=3000
)
env.close()

# Visualizar progreso de entrenamient0
manager.plot_training_history(id_1)

# DynaQAgent

In [ ]:
from dyna_q_agent import DynaQAgent

In [ ]:
# Crear agente con configuración inicial
agent = DynaQAgent(x_bins=100, vel_bins=100, action_bins=20)

# Entrenar el agente
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Dyna-Q...")
agent.train_agent(
    env, 
    episodes=5000,
    alpha=0.03,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.2,
    max_steps=1000,
    planning_steps=20
)
env.close()

# Guardar el modelo entrenado
agent.save_model('dyna_agent_trained.npy')

# Visualizar progreso de entrenamiento
agent.plot_training_history()

In [ ]:
# Cargar un modelo Dyna-Q guardado y evaluarlo
agent = DynaQAgent(x_bins=100, vel_bins=100, action_bins=20)
agent.load_model('dyna_agent_trained.npy')

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Evaluando el modelo cargado...")
test_rewards, test_steps, success_count = agent.test_agent(env, episodes=5, render=False)
env.close()

print(f"\nExitos en testing: {success_count}/5")
print(f"Promedio de rewards: {np.mean(test_rewards):.2f}")
print(f"Promedio de steps: {np.mean(test_steps):.2f}")

# Q-Learning variando el epsilon

In [ ]:
from q_learning_agent import QLearningAgent

# Crear agente con configuración inicial
agent = QLearningAgent(x_bins=100, vel_bins=100, action_bins=20)
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')

In [ ]:
print("Entrenando agente Q-Learning...")
id_e1 = "8k_episodes_eps075"
manager.train_run(
    env, 
    run_id=id_e1,
    x_bins=100,
    vel_bins=100,
    action_bins=20,
    episodes=8000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.075,
    max_steps=3000,
)
env.close()
manager.plot_training_history(id_e1)

In [ ]:
# Entrenar el agente
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Q-Learning...")
id_e2 = "8k_episodes_eps025"
manager.train_run(
    env, 
    run_id=id_e2,
    x_bins=100,
    vel_bins=100,
    action_bins=20,
    episodes=8000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.025,
    max_steps=3000,
)
env.close()
manager.plot_training_history(id_e2)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Q-Learning...")
id_e3 = "8k_episodes_eps001"
manager.train_run(
    env, 
    run_id=id_e3,
    x_bins=100,
    vel_bins=100,    
    action_bins=20,
    episodes=8000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_e3)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Q-Learning...")
id_e4 = "5500_episodes_eps0005"
manager.train_run(
    env, 
    run_id=id_e4,
    x_bins=100,
    vel_bins=100,
    action_bins=20,
    episodes=5500,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.0005,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_e4)

Variando el epsilon start

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Q-Learning...")
id_es1= "8k_episodes_eps075"
manager.train_run(
    env, 
    run_id=id_es1,
    x_bins=100,
    vel_bins=100,
    action_bins=20,
    episodes=8000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.75, 
    epsilon_end=0.025,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_es1)

Comparacion de graficas

In [ ]:
manager.plot_comparisons()

# Analsis para mejorar dicretización

## Q-learning

### Setup

In [ ]:
import gymnasium as gym
from q_experiments_manager import ExperimentManager
from q_learning_agent import QLearningAgent
from discretization_evaluator import DiscretizationEvaluator

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
manager = ExperimentManager()

In [ ]:
evaluator = DiscretizationEvaluator(manager)

train_kwargs = {
    'episodes': 5000,
    'alpha': 0.05,
    'gamma': 0.99,
    'epsilon_start': 0.99,
    'epsilon_end': 0.0005,
    'max_steps': 3000,
}

### Pruebas variando los action Bins

In [ ]:
evaluator.evaluate_group(
    env=env,
    x_bins_list=[50],
    vel_bins_list=[50],
    action_bins_list=[5, 10, 20],  
    train_kwargs=train_kwargs,
    test_episodes=10,
    test_max_steps=3000,
    prefix="G1_Acciones"
)

In [ ]:
evaluator.plot_ranking()

### Variaciones con mejor resultado anterior

In [ ]:
evaluator.evaluate_group(
    env=env,
    x_bins_list=[30, 100],      # Baja y alta resolución de posición
    vel_bins_list=[20, 100],     # Baja y alta resolución de velocidad
    action_bins_list=[10],       # Fijamos el ganador tentativo del grupo anterior
    train_kwargs=train_kwargs,
    test_episodes=10,
    test_max_steps=3000,
    prefix="G2_Estados"
)

env.close()

In [ ]:
print("\n*** PROCESANDO RESULTADOS DE LOS GRUPOS EJECUTADOS ***")
evaluator.print_summary()

## Gráficas

In [ ]:
evaluator.plot_heatmap(metric='avg_reward')
evaluator.plot_pareto(metric='avg_reward')
evaluator.plot_convergence(top_n=3)

# Con los valores resultantes

30, 20 y 10

In [ ]:
print("Entrenando agente Q-Learning...")
id_a1 = "4k_episodes_eps015"
manager.train_run(
    env=env, 
    run_id=id_a1,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.0005,
    max_steps=3000
)
env.close()

# Visualizar progreso de entrenamient0
manager.plot_training_history(id_a1)

## Dyna Q con los valores

In [ ]:
from dyna_q_experiments_manager import DynaQExperimentManager

dyna_manager = DynaQExperimentManager()

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d1 = "dyna_4k_episodes_eps015"
dyna_manager.train_run(
    env=env,
    run_id=id_d1,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=30,
    episodes=4000,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d1)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d1 = "dyna_2.5k_episodes_planning30"
dyna_manager.train_run(
    env=env,
    run_id=id_d1,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=30,
    episodes=2500,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d1)

#### Bajando los planning steps

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d3 = "dyna_2.5k_episodes_planning15"
dyna_manager.train_run(
    env=env,
    run_id=id_d3,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=15,
    episodes=2500,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d3)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d3 = "dyna_2.5k_episodes_planning5"
dyna_manager.train_run(
    env=env,
    run_id=id_d3,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=5,
    episodes=2500,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d3)

In [ ]:
dyna_manager.plot_comparisons()

Con el mejor resultado menor cantidad de episodios

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d5 = "dyna_2k_episodes_planning5"
dyna_manager.train_run(
    env=env,
    run_id=id_d5,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=5,
    episodes=2000,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d5)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d6 = "dyna_1.6k_episodes_planning5"
dyna_manager.train_run(
    env=env,
    run_id=id_d6,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=5,
    episodes=1600,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d6)

menos episodios y subir planinng step

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d7 = "dyna_1k_episodes_planning15"
dyna_manager.train_run(
    env=env,
    run_id=id_d7,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=15,
    episodes=1000,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d7)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d8 = "dyna_800_episodes_planning15"
dyna_manager.train_run(
    env=env,
    run_id=id_d8,
    x_bins=30, vel_bins=20, action_bins=10,
    planning_steps=15,
    episodes=800,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d8)

In [ ]:
dyna_manager.plot_comparisons()

## Como funciona sin mejora de discretizacion

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d9 = "dynaog_2k_episodes_planning5"
dyna_manager.train_run(
    env=env,
    run_id=id_d9,
    x_bins=100, vel_bins=100, action_bins=20,
    planning_steps=5,
    episodes=2000,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d9)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d10 = "dynaog_1.5k_episodes_planning15"
dyna_manager.train_run(
    env=env,
    run_id=id_d10,
    x_bins=100, vel_bins=100, action_bins=20,
    planning_steps=15,
    episodes=1500,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000,
)
env.close()
dyna_manager.plot_training_history(id_d10)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d11 = "dynaog_1k_episodes_planning15"
dyna_manager.train_run(
    env=env,
    run_id=id_d11,
    x_bins=100, vel_bins=100, action_bins=20,
    planning_steps=15,
    episodes=1000,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=5000,
)
env.close()
dyna_manager.plot_training_history(id_d11)

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
id_d12 = "dynaog_1k_episodes_planning15"
dyna_manager.train_run(
    env=env,
    run_id=id_d12,
    x_bins=100, vel_bins=100, action_bins=20,
    planning_steps=15,
    episodes=400,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=5000,
)
env.close()
dyna_manager.plot_training_history(id_d12)

In [ ]:
dyna_manager.plot_comparisons()

# Evaluancion de rendimiento

In [ ]:
from hyperparameter_evaluator import HyperparameterEvaluator
from subprocess import run

ql_eval   = HyperparameterEvaluator(manager)       
dyna_eval = HyperparameterEvaluator(dyna_manager)  

## Q-learning

### Epsilon End

In [ ]:
id_qe1 = "ql_eps_001"
id_qe2 = "ql_eps_01"
id_qe3 = "ql_eps_05"

In [ ]:

print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qe1,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qe1)

In [ ]:
print("Entrenando agente Q-Learning...")
manager.train_run(
    env, 
    run_id=id_qe2,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99,
    epsilon_end=0.01,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qe2)

In [ ]:
print("Entrenando agente Q-Learning...")
manager.train_run(
    env, 
    run_id=id_qe3,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99,
    epsilon_end=0.05,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qe3)

### Evaluación

In [ ]:
ql_eval.plot_convergence_curves(run_ids=[id_qe1, id_qe2, id_qe3])

In [ ]:

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
ql_eval.evaluate(env, run_ids=[id_qe1, id_qe2, id_qe3], episodes=10)
env.close()

ql_eval.plot_ranking(run_ids=[id_qe1, id_qe2, id_qe3])

In [ ]:
ql_eval.print_full_report(run_ids=[id_qe1, id_qe2, id_qe3])

### Alpha

In [ ]:
id_qa1= "ql_alpha_001"
id_qa2= "ql_alpha_005"
id_qa3= "ql_alpha_01"

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qa1,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.01,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qa1)

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qa2,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qa2)

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qa3,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.1,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qa3)

### Evaluación

In [ ]:
ql_eval.plot_convergence_curves(run_ids=[id_qa1, id_qa2, id_qa3])

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
ql_eval.evaluate(env, run_ids=[id_qa1, id_qa2, id_qa3], episodes=10)
env.close()

ql_eval.plot_ranking(run_ids=[id_qa1, id_qa2, id_qa3])

In [ ]:
ql_eval.print_full_report(run_ids=[id_qa1, id_qa2, id_qa3])

### Action Bins

In [ ]:
id_qab1="ql_ab5"
id_qab2="ql_ab10"
id_qab3="ql_ab15"
id_qab4="ql_ab20"

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qab1,
    x_bins=30,         
    vel_bins=20,       
    action_bins=5,    
    episodes=4000,
    alpha=0.1,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qab1)

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qab2,
    x_bins=30,         
    vel_bins=20,       
    action_bins=10,    
    episodes=4000,
    alpha=0.1,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qab2)

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qab3,
    x_bins=30,         
    vel_bins=20,       
    action_bins=15,    
    episodes=4000,
    alpha=0.1,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qab3)

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qab4,
    x_bins=30,         
    vel_bins=20,       
    action_bins=20,    
    episodes=4000,
    alpha=0.1,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qab4)

### Evaluación

In [ ]:
ql_eval.plot_convergence_curves(run_ids=[id_qab1, id_qab2, id_qab3, id_qab4])

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
ql_eval.evaluate(env, run_ids=[id_qab1, id_qab2, id_qab3, id_qab4], episodes=10)
env.close()

ql_eval.plot_ranking(run_ids=[id_qab1, id_qab2, id_qab3, id_qab4])

In [ ]:
ql_eval.print_full_report(run_ids=[id_qab1, id_qab2, id_qab3, id_qab4])

### Con mayor resolusion/granularidad

In [ ]:
id_qb1="ql_bins_30x20"
id_qb2="ql_bins_100x100"

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qb1,
    x_bins=30,         
    vel_bins=20,       
    action_bins=20,    
    episodes=4000,
    alpha=0.1,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qb1)

In [ ]:
print("Entrenando agente Q-Learning...")

manager.train_run(
    env=env, 
    run_id=id_qb2,
    x_bins=100,         
    vel_bins=100,       
    action_bins=20,    
    episodes=4000,
    alpha=0.1,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.001,
    max_steps=3000
)
env.close()
manager.plot_training_history(id_qb2)

### Evaluación

In [ ]:
ql_eval.plot_convergence_curves(run_ids=[id_qb1, id_qb2])

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
ql_eval.evaluate(env, run_ids=[id_qb1, id_qb2], episodes=10)
env.close()

ql_eval.plot_ranking(run_ids=[id_qb1, id_qb2])

In [ ]:
ql_eval.print_full_report(run_ids=[id_qb1, id_qb2])

## Seleccion de DynaQ

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
dyna_eval.evaluate(env, episodes=10)
env.close()

In [ ]:
dyna_eval.print_best()

In [ ]:
dyna_eval.plot_ranking()

## Comparar ambos

In [ ]:
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
ql_eval.evaluate(env, episodes=10)
dyna_eval.evaluate(env, episodes=10)
env.close()

import matplotlib.pyplot as plt

todos = {**ql_eval.eval_results, **dyna_eval.eval_results}
names = list(todos.keys())
rewards = [todos[n]['avg_reward'] for n in names]

plt.barh(names, rewards)
plt.xlabel('avg_reward')
plt.title('Q-Learning vs Dyna-Q')
plt.tight_layout()
plt.show()

In [ ]:
import importlib
import hyperparameter_evaluator
importlib.reload(hyperparameter_evaluator)
from hyperparameter_evaluator import HyperparameterEvaluator

ql_eval = HyperparameterEvaluator(manager)
dyna_eval = HyperparameterEvaluator(dyna_manager)

In [ ]:
# Evaluar ambos por separado (una sola vez)
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
ql_eval.evaluate(env, episodes=10)
dyna_eval.evaluate(env, episodes=10)
env.close()

# Crear el evaluador combinado
combined = ql_eval.merge(dyna_eval)


combined.plot_ranking()
combined.plot_reward_vs_steps()
combined.plot_convergence_curves()
combined.plot_tradeoff()
combined.print_full_report()

In [ ]:
ql_eval.plot_ranking()   # todos los Q-Learning
dyna_eval.plot_ranking() # todos los Dyna-Q

# Guardado de modelos entrenados para entrega

In [ ]:
agent = QLearningAgent(x_bins=30, vel_bins=20, action_bins=20)

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
agent.train_agent(
    env=env,   
    episodes=4000,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.05,
    max_steps=3000
)
env.close()

agent.save_model('ql_agent_trained_final.npy')

agent.plot_training_history()


from dyna_q_agent import DynaQAgent

agent = DynaQAgent(x_bins=30, vel_bins=20, action_bins=10)

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Dyna-Q...")
agent.train_agent(
    env=env,
    planning_steps=5,
    episodes=2500,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000
)
env.close()

agent.save_model('dyna_agent_trained_final.npy')

agent.plot_training_history()

In [ ]:
agent = QLearningAgent(x_bins=30, vel_bins=20, action_bins=20)
agent.load_model('ql_agent_trained_final.npy')

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
agent.train_agent(
    env=env,   
    episodes=4000,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.05,
    max_steps=3000
)
env.close()

agent.save_model('ql_agent_trained_final.npy')

agent.plot_training_history()

agent = DynaQAgent(x_bins=30, vel_bins=20, action_bins=10)
agent.load_model('dyna_agent_trained_final.npy')

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
agent.train_agent(
    env=env,
    planning_steps=5,
    episodes=2500,
    alpha=0.05,
    gamma=0.99,
    epsilon_start=0.99,
    epsilon_end=0.0005,
    max_steps=3000
)
env.close()

agent.save_model('dyna_agent_trained_final.npy')

agent.plot_training_history()


In [ ]:
agent.load_model('ql_agent_trained_final.npy')

env = gym.make('MountainCarContinuous-v0', render_mode='human')
agent.test_agent(env, episodes=3, render=True)
env.close()

agent.load_model('dyna_agent_trained_final.npy')

env = gym.make('MountainCarContinuous-v0', render_mode='human')
agent.test_agent(env, episodes=3, render=True)
env.close()

# Pruebas Anteriores

In [ ]:
from q_learning_agent import QLearningAgent

# Crear agente con configuración inicial
agent = QLearningAgent(x_bins=100, vel_bins=100, action_bins=20)

# Entrenar el agente
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Q-Learning...")
agent.train_agent(
    env, 
    episodes=8000,
    alpha=0.05,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.15,
    max_steps=3000
)
env.close()

# Visualizar progreso de entrenamiento
agent.plot_training_history()

In [5]:
from dyna_q_agent import DynaQAgent

agent = DynaQAgent(x_bins=100, vel_bins=100, action_bins=20)

env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
print("Entrenando agente Dyna-Q...")
agent.train_agent(
    env, 
    episodes=5000,
    alpha=0.03,
    gamma=0.99, 
    epsilon_start=0.99, 
    epsilon_end=0.2,
    max_steps=1000,
    planning_steps=20
)
env.close()

# Guardar el modelo entrenado
agent.save_model('dyna_agent_trained.npy')

# Visualizar progreso de entrenamiento
agent.plot_training_history()

Entrenando agente Dyna-Q...


KeyboardInterrupt: 

In [ ]:
# Diagnóstico rápido del modelo cargado
import numpy as np

from q_learning_agent import QLearningAgent

# Crear agente con configuración inicial
agent = QLearningAgent(x_bins=100, vel_bins=100, action_bins=20)
agent.load_model('dyna_agent_trained.npy')

print('Q shape:', getattr(agent, 'Q', None).shape if hasattr(agent, 'Q') else 'no Q')
print('Q sum:', np.sum(agent.Q))
print('Q max:', np.max(agent.Q))
print('Non-zero count:', np.count_nonzero(agent.Q))

# Mostrar los valores de Q para el estado inicial
env = gym.make('MountainCarContinuous-v0', render_mode='rgb_array')
obs, _ = env.reset()
state = agent.discretize_state(obs)
print('Initial observation:', obs)
print('Discretized state:', state)
print('Q[state]:', agent.Q[state])

# Acción determinista según la política actual
action = agent.next_action(state, epsilon=0.0, training=False)
print('Selected action (deterministic):', action)
env.close()